Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Create dataframe

In [ ]:
df = pd.read_csv('/content/fact_saless.csv')

In [ ]:
df.head()

,DateKey,ProductKey,BranchKey,Sales_Quantity,Unit_Price,Unit_Cost,Revenue,Total_Cost,Net_Profit,Profit_Margin,Expired_Quantity,Damaged_Quantity,Waste_Quantity,Waste_Rate
0,20210101,1,1,24,89.82,58.64,2155.68,1407.36,748.32,0.347139,0,0,0,0.000000
1,20210101,2,1,135,100.94,67.55,13626.90,9119.25,4305.00,0.315919,2,1,3,0.022222
2,20210101,3,1,552,35.43,20.77,19557.36,11465.04,8030.01,0.410588,0,3,3,0.005435
3,20210101,4,1,38,29.24,21.71,1111.12,824.98,286.14,0.257524,0,0,0,0.000000
4,20210101,5,1,20,87.09,57.26,1741.80,1145.20,596.60,0.342519,0,0,0,0.000000


In [ ]:
def risk_level(row):
  score = 0
  if row["Waste_Rate"] >= 0.10:
    score += 2
  if row["Expired_Quantity"] >= 3:
    score += 2
  if row["Damaged_Quantity"] >= 2:
    score += 1
  if row["Profit_Margin"] < 0.20:
    score += 1

  if score >= 4:
    return "High Risk"
  elif score >= 2:
    return "Medium Risk"
  else:
    return "Low Risk"
df["Risk_Level"] = df.apply(risk_level, axis=1)


In [ ]:
df["Risk_Level"].value_counts()

,count
Risk_Level,
Low Risk,755445
Medium Risk,33663
High Risk,32142


In [ ]:
features = ["ProductKey", "BranchKey", "Sales_Quantity", "Revenue", "Total_Cost"]


In [ ]:
X = df[features]
Y = df["Risk_Level"]

In [ ]:
encoder = LabelEncoder()
Y = encoder.fit_transform(Y)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, Y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [ ]:
predictions = model.predict(X_test)

In [ ]:
acc = accuracy_score(Y_test, predictions)
print(f"Accuracy: {acc}")

Accuracy: 0.9849741248097412


In [ ]:
print(classification_report(Y_test, predictions, target_names=encoder.classes_))

              precision    recall  f1-score   support

   High Risk       0.85      0.85      0.85      6388
    Low Risk       1.00      1.00      1.00    151196
 Medium Risk       0.84      0.84      0.84      6666

    accuracy                           0.98    164250
   macro avg       0.90      0.90      0.90    164250
weighted avg       0.99      0.98      0.98    164250



In [ ]:
conmat = confusion_matrix(Y_test, predictions)
print(conmat)

[[  5459    182    747]
 [   166 150693    337]
 [   767    269   5630]]


In [ ]:
df.head(50)

,DateKey,ProductKey,BranchKey,Sales_Quantity,Unit_Price,Unit_Cost,Revenue,Total_Cost,Net_Profit,Profit_Margin,Expired_Quantity,Damaged_Quantity,Waste_Quantity,Waste_Rate,Risk_Level
0,20210101,1,1,24,89.82,58.64,2155.68,1407.36,748.32,0.347139,0,0,0,0.000000,Low Risk
1,20210101,2,1,135,100.94,67.55,13626.90,9119.25,4305.00,0.315919,2,1,3,0.022222,Low Risk
2,20210101,3,1,552,35.43,20.77,19557.36,11465.04,8030.01,0.410588,0,3,3,0.005435,Low Risk
3,20210101,4,1,38,29.24,21.71,1111.12,824.98,286.14,0.257524,0,0,0,0.000000,Low Risk
4,20210101,5,1,20,87.09,57.26,1741.80,1145.20,596.60,0.342519,0,0,0,0.000000,Low Risk
5,20210101,6,1,11,164.52,113.49,1809.72,1248.39,561.33,0.310175,0,0,0,0.000000,Low Risk
6,20210101,7,1,27,67.99,44.84,1835.73,1210.68,400.85,0.218360,5,0,5,0.185185,High Risk
7,20210101,8,1,22,102.22,70.71,2248.84,1555.62,693.22,0.308257,0,0,0,0.000000,Low Risk
8,20210101,9,1,21,130.89,87.52,2748.69,1837.92,910.77,0.331347,0,0,0,0.000000,Low Risk
9,20210101,10,1,14,53.69,40.14,751.66,561.96,189.70,0.252375,0,0,0,0.000000,Low Risk


In [ ]:
import ipywidgets as widgets
from IPython.display import display

product = widgets.IntText(description='Product')
branch = widgets.IntText(description='Branch')
sales = widgets.FloatText(description='Sales')
revenue = widgets.FloatText(description='Revenue')
cost = widgets.FloatText(description='Cost')

display(product, branch, sales, revenue, cost)
button = widgets.Button(description="Predict")

output = widgets.Output()

def predict(b):
    with output:
        output.clear_output()

        user = pd.DataFrame({
            "ProductKey":[product.value],
            "BranchKey":[branch.value],
            "Sales_Quantity":[sales.value],
            "Revenue":[revenue.value],
            "Total_Cost":[cost.value]
        })

        pred = model.predict(user)

        risk = encoder.inverse_transform(pred)

        print("Predicted Risk Level:", risk[0])

button.on_click(predict)

display(button, output)

IntText(value=0, description='Product')

IntText(value=0, description='Branch')

FloatText(value=0.0, description='Sales')

FloatText(value=0.0, description='Revenue')

FloatText(value=0.0, description='Cost')

Button(description='Predict', style=ButtonStyle())

Output()